# Entrenamiento M2 (YOLO11-seg) y M3 (YOLO11-pose) con Roboflow
- **M2**: segmentación — `motocicleta, automovil, minibus, camion, bus`
- **M3**: keypoints — clase `rueda`, **1 keypoint** (centro de la rueda)

Colab: *Entorno de ejecución → GPU T4*. La API key se pide con `getpass` (no se guarda en el notebook).

In [ ]:
!pip install -q ultralytics roboflow
import torch, ultralytics, sys
print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), ultralytics.__version__)

In [ ]:
from getpass import getpass
from roboflow import Roboflow
API_KEY = getpass("Roboflow API key: ")
rf = Roboflow(api_key=API_KEY)
WS = "luzmilas-workspace"
VER_M2, VER_M3 = 1, 1   # <- cambia al numero de version que generaste en Roboflow
ds2 = rf.workspace(WS).project("m2-qfggl").version(VER_M2).download("yolov11", location="/content/ds_m2")
ds3 = rf.workspace(WS).project("m3-r2mey").version(VER_M3).download("yolov11", location="/content/ds_m3")
!cat /content/ds_m2/data.yaml
!cat /content/ds_m3/data.yaml

## Datasets extra de Roboflow Universe (opcional, recomendado)
Busca en Universe datasets de **vehículos (segmentation)** y de **ruedas (keypoints, 1 punto)**, y agrégalos abajo.
`mapeo` = nombre de clase del dataset extra → clase destino. Las clases no mapeadas se ignoran.
Para M3 el extra debe tener el mismo formato de 1 keypoint; si solo tiene cajas, no lo uses.

In [ ]:
import shutil, yaml, random
from pathlib import Path

CLASES_M2 = ["motocicleta","automovil","minibus","camion","bus"]

# Ejemplo (descomenta y ajusta workspace/proyecto/version/mapeo):
EXTRAS_M2 = [
    # {"ws":"ESPACIO","proj":"PROYECTO","ver":1,
    #  "mapeo":{"motorcycle":"motocicleta","car":"automovil","van":"minibus","truck":"camion","bus":"bus"}},
]
EXTRAS_M3 = [
    # {"ws":"ESPACIO","proj":"PROYECTO","ver":1,"mapeo":{"wheel":"rueda"}},
]

def agregar_extra(dest, destino_clases, e, n):
    d = Path(f"/content/extra_{n}")
    r = rf.workspace(e["ws"]).project(e["proj"]).version(e["ver"]).download("yolov11", location=str(d))
    names = yaml.safe_load(open(d/"data.yaml"))["names"]
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    idmap = {i: destino_clases.index(e["mapeo"][c.lower()]) for i, c in enumerate(names) if c.lower() in e["mapeo"]}
    cont = 0
    for split in ("train",):   # extras solo aportan a train; valid/test quedan con tus imagenes propias
        for img in (d/split/"images").glob("*"):
            lab = d/split/"labels"/(img.stem+".txt")
            if not lab.exists(): continue
            out = []
            for ln in lab.read_text().splitlines():
                p = ln.split()
                if p and int(p[0]) in idmap:
                    p[0] = str(idmap[int(p[0])]); out.append(" ".join(p))
            if out:
                (dest/"train"/"images").mkdir(parents=True, exist_ok=True); (dest/"train"/"labels").mkdir(parents=True, exist_ok=True)
                shutil.copy2(img, dest/"train"/"images"/f"x{n}_{img.name}")
                (dest/"train"/"labels"/f"x{n}_{img.stem}.txt").write_text("\n".join(out))
                cont += 1
    print(f"extra {e['proj']}: {cont} imagenes agregadas")

for n, e in enumerate(EXTRAS_M2): agregar_extra(Path("/content/ds_m2"), CLASES_M2, e, f"m2_{n}")
for n, e in enumerate(EXTRAS_M3): agregar_extra(Path("/content/ds_m3"), ["rueda"], e, f"m3_{n}")

for nom in ("ds_m2","ds_m3"):
    for s in ("train","valid","test"):
        p = Path(f"/content/{nom}/{s}/images")
        print(nom, s, len(list(p.glob('*'))) if p.exists() else 0)

## Entrenamiento

In [ ]:
from ultralytics import YOLO
m2 = YOLO("yolo11s-seg.pt")
m2.train(data="/content/ds_m2/data.yaml", epochs=100, imgsz=640, batch=16, patience=25,
         fliplr=0.5, hsv_v=0.4, degrees=5, mosaic=1.0, project="/content/runs", name="m2", exist_ok=True)

In [ ]:
m3 = YOLO("yolo11s-pose.pt")
m3.train(data="/content/ds_m3/data.yaml", epochs=120, imgsz=640, batch=16, patience=30,
         fliplr=0.0, hsv_v=0.4, degrees=3, project="/content/runs", name="m3", exist_ok=True)

## Métricas (para el informe)

In [ ]:
r2 = YOLO("/content/runs/m2/weights/best.pt").val(data="/content/ds_m2/data.yaml", split="test" if Path("/content/ds_m2/test/images").exists() else "val")
print("M2 mask mAP50:", r2.seg.map50, " mAP50-95:", r2.seg.map, " box mAP50:", r2.box.map50)
r3 = YOLO("/content/runs/m3/weights/best.pt").val(data="/content/ds_m3/data.yaml", split="test" if Path("/content/ds_m3/test/images").exists() else "val")
print("M3 pose mAP50:", r3.pose.map50, " mAP50-95:", r3.pose.map, " box mAP50:", r3.box.map50)

## Guardar pesos en Drive (`m2_seg.pt` y `m3_pose.pt`)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
D = Path("/content/drive/MyDrive/peaje_pesos"); D.mkdir(parents=True, exist_ok=True)
shutil.copy2("/content/runs/m2/weights/best.pt", D/"m2_seg.pt")
shutil.copy2("/content/runs/m3/weights/best.pt", D/"m3_pose.pt")
shutil.copytree("/content/runs/m2", D/"resultados_m2", dirs_exist_ok=True)
shutil.copytree("/content/runs/m3", D/"resultados_m3", dirs_exist_ok=True)
print(list(D.iterdir()))